# Map LA's air quality in an afternoon

**Terrain beginner · Python project · EPA AirNow**

Pull air-quality readings for the Los Angeles area and map monitoring sites color-coded by AQI.

---

## Learning objectives

By the end, you should be able to:

- Load environmental data from a JSON file or a live API
- Clean and summarize a table with pandas
- Map point data with color meaning (AQI categories)
- Write a short, evidence-based takeaway

**Time:** ~45 min · **Dataset:** [EPA AirNow explainer](../explainer.html?id=epa-airnow)

---

## For mentors

- **Prerequisites:** None, but you need a free [AirNow API key](https://docs.airnowapi.org/account/request/) for live data.
- **Colab:** Add your key under **Secrets** as `AIRNOW_API_KEY`, or paste it in the notebook cell.
- **Watch for:** Students confusing AQI (index) with raw pollutant concentrations.

**Suggested flow:** Read each section aloud briefly → student runs the cell → use the *Mentor check-in* question before moving on. Do not skip the final takeaway.

---


**Goal:** Import the Python tools we need.

**Concept:** `pandas` handles tables. `folium` draws interactive maps. `requests` calls the AirNow API when you go live.

**Run the cell below**, then compare your output to what we expect.

**You should see:** No output — just imports (Colab may pip-install folium once).

**Mentor check-in:** Ask: *Which of these libraries have you seen before?*

**If something breaks:** If `folium` fails, re-run the cell — the install line runs automatically.


In [ ]:
# Run once if packages are missing (Colab usually has these)
try:
    import folium
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
    import folium

import json
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

**Goal:** Get air-quality observations into a table.

**Concept:** AirNow publishes **live** readings from ground monitors. You need a free API key (link in Section 1 intro).

**Run the cell below**, then compare your output to what we expect.

**You should see:** `Fetched N live observations from AirNow` and a preview with `SiteName`, `AQI`, `Latitude`.

**Mentor check-in:** Ask: *What does each row represent — a person, a city, or a monitor?*

**If something breaks:** Check your API key. In Colab: **Secrets** → add `AIRNOW_API_KEY`.


In [ ]:
import os
from pathlib import Path

def get_airnow_api_key():
    key = os.environ.get("AIRNOW_API_KEY", "").strip()
    if key:
        return key
    for env_path in [Path(".env"), Path("../.env")]:
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                if line.startswith("AIRNOW_API_KEY="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    try:
        from google.colab import userdata
        return userdata.get("AIRNOW_API_KEY").strip()
    except Exception:
        return ""

AIRNOW_API_KEY = get_airnow_api_key() or ""  # or paste key here temporarily

def fetch_airnow_live(api_key, lat=34.05, lon=-118.25, distance=75):
    url = "https://www.airnowapi.org/aq/observation/latLong/current/"
    params = {
        "format": "application/json",
        "latitude": lat,
        "longitude": lon,
        "distance": distance,
        "API_KEY": api_key,
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

if not AIRNOW_API_KEY:
    raise ValueError(
        "AirNow requires a free API key. Get one at https://docs.airnowapi.org/account/request/ "
        "then set AIRNOW_API_KEY in Colab Secrets, a .env file, or paste it in this cell."
    )

records = fetch_airnow_live(AIRNOW_API_KEY)
print(f"Fetched {len(records)} live observations from EPA AirNow")

df = pd.DataFrame(records)
df.head()

**Goal:** Prepare one clean row per monitoring site.

**Concept:** We convert AQI to numbers, drop bad coordinates, and keep the highest reading if a site appears twice. Then we compute quick summary stats mentors can discuss.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Printed site count, median AQI, and which site is highest — plus a sorted table of the top sites.

**Mentor check-in:** Ask: *Why take the max AQI if there are duplicates?* (Answer: conservative snapshot of worst pollutant at that site.)


In [ ]:
clean = (
    df.assign(AQI=pd.to_numeric(df["AQI"], errors="coerce"))
      .dropna(subset=["Latitude", "Longitude", "AQI"])
      .sort_values("AQI", ascending=False)
      .drop_duplicates(subset=["SiteName"], keep="first")
      .rename(columns={"Latitude": "lat", "Longitude": "lon"})
)

print(f"Monitoring sites: {len(clean)}")
print(f"Median AQI: {clean['AQI'].median():.0f}")
print(f"Highest AQI: {clean['AQI'].max():.0f} at {clean.iloc[0]['SiteName']}")
clean[["SiteName", "ReportingArea", "ParameterName", "AQI", "Category"]].head(8)

**Goal:** Assign colors that match EPA AQI categories.

**Concept:** Color is a design choice, but EPA uses standard breakpoints: Good ≤50, Moderate ≤100, etc. We mirror Terrain's palette.

**Run the cell below**, then compare your output to what we expect.

**You should see:** No visible output — we add a `color` column to the table.

**Mentor check-in:** Ask: *If AQI is 62, which category is that?*


In [ ]:
def aqi_color(aqi):
    if aqi <= 50:
        return "#2F6B4F"   # Good
    if aqi <= 100:
        return "#D98E5A"   # Moderate
    if aqi <= 150:
        return "#C45C5C"   # Unhealthy for sensitive groups
    return "#7B2D26"       # Unhealthy+

clean = clean.assign(color=clean["AQI"].map(aqi_color))
clean[["SiteName", "AQI", "Category", "color"]].head()

**Goal:** Build the interactive map — your main deliverable.

**Concept:** Each monitor becomes a circle. Pop-ups show site name, AQI, and pollutant. Pan around LA and click a few sites together.

**Run the cell below**, then compare your output to what we expect.

**You should see:** An interactive map centered on LA with colored dots.

**Mentor check-in:** Ask: *Do higher-AQI sites cluster anywhere? Near freeways, inland valleys, coast?*

**If something breaks:** If the map is blank, confirm `lat`/`lon` columns exist and AQI is numeric.


In [ ]:
m = folium.Map(location=[34.05, -118.25], zoom_start=9, tiles="CartoDB positron")

for _, row in clean.iterrows():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=9,
        color=row.color,
        fill=True,
        fill_color=row.color,
        fill_opacity=0.85,
        popup=f"<b>{row.SiteName}</b><br>AQI {int(row.AQI)} ({row.Category})<br>{row.ParameterName}",
    ).add_to(m)

m

**Goal:** Optional: a static bar chart for slides or reports.

**Concept:** Maps tell spatial stories; bar charts compare sites directly. Good for a one-slide summary.

**Run the cell below**, then compare your output to what we expect.

**You should see:** Horizontal bar chart of AQI by site name.

**Mentor check-in:** Ask: *When would you choose a map vs. a bar chart for the same data?*


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(clean["SiteName"], clean["AQI"], color=clean["color"])
ax.set_xlabel("AQI")
ax.set_title("Current AQI by monitoring site (LA region)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---

## Final takeaway (student writes; mentor reviews)

Have the student answer in **3–4 sentences** in their own words:

1. Which areas show the highest AQI in this snapshot?
2. Is ozone or PM2.5 driving the peak readings? (Check `ParameterName`.)
3. What limitation should readers keep in mind about monitor spacing?

**Mentor rubric (quick):**
- ✅ Names specific places or sites from the data
- ✅ Uses AQI vocabulary correctly
- ✅ Mentions at least one limitation (gaps between monitors, single-day snapshot, etc.)

**Next steps:** Compare with long-term burden in [CalEnviroScreen (R)](../notebooks/calenviroscreen_burden.Rmd) or browse [Terrain datasets](../datasets.html).
